<a href="https://colab.research.google.com/github/rithika-d/Big_Data_Analytics_Midterm_Project/blob/main/Big_Data_Analytics_Midterm2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 141.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

!pip install unsloth

https://huggingface.co/0llheaven/Llama-3.2-11B-Vision-Radiology-mini

In [ ]:
from unsloth import FastVisionModel
from PIL import Image
import torch

# Load model and tokenizer/processor
llm_model, tokenizer = FastVisionModel.from_pretrained(
    "0llheaven/Llama-3.2-11B-Vision-Radiology-mini",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
FastVisionModel.for_inference(llm_model)

def _model_input_device(model):
    # For 4bit/8bit, model is often sharded; pick the device of the embedding / first param
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cuda")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Mllama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
from PIL import Image
import torch, gc

def load_image_small(path, size=448):
    img = Image.open(path).convert("RGB")
    img.thumbnail((size, size), Image.Resampling.LANCZOS)
    return img

def predict_radiology_description(image, instruction):
    messages = [{
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": instruction}],
    }]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

    inputs = tokenizer(
        text=prompt,
        images=image,
        add_special_tokens=False,
        return_tensors="pt",
    )

    dev = next(llm_model.parameters()).device
    inputs = {k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.no_grad():
        out = llm_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
            use_cache=False,   # biggest VRAM saver
        )

    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

gc.collect()
torch.cuda.empty_cache()

image = load_image_small("/content/drive/MyDrive/chest_xray/train/PNEUMONIA/person1000_bacteria_2931.jpeg", size=448)  # if OOM -> 336
instruction = "Write FINDINGS and IMPRESSION. Be concise. If unsure, say so."
print(predict_radiology_description(image, instruction))

user

Write FINDINGS and IMPRESSION. Be concise. If unsure, say so.assistant

**FINDINGS:**

* The radiographic image shows a right-sided pneumothorax with mediastinal shift to the left.
* There is an endotracheal tube in place, and a nasogastric tube is also visible.
* The right lung is collapsed, and the diaphragm is elevated.

**IMPRESSION:**

* The patient has a right-sided pneumothorax with significant mediastinal shift, indicating a moderate to severe lung collapse.
* The presence of an endotracheal tube and nasogastric tube suggests that the patient is receiving respiratory and nutritional support.
* Further evaluation and


!pip -q install -U "timm>=0.9.8"

In [ ]:
image = load_image_small("/content/drive/MyDrive/chest_xray/train/NORMAL/IM-0115-0001.jpeg", size=448)  # if OOM -> 336
instruction = "Write FINDINGS and IMPRESSION. Be concise. If unsure, say so."
print(predict_radiology_description(image, instruction))

user

Write FINDINGS and IMPRESSION. Be concise. If unsure, say so.assistant

**Findings:**

* The right clavicle is not visible, suggesting a possible fracture or dislocation.
* The right humerus is also not visible, which could indicate a fracture or dislocation.
* The right scapula appears normal.
* The right arm is rotated internally, which may be due to a fracture or dislocation.

**Impression:**

The radiograph suggests a possible fracture or dislocation of the right clavicle and humerus, but further evaluation and imaging studies are needed to confirm the diagnosis.


In [ ]:
import importlib
import eva_x
importlib.reload(eva_x)
print("eva_x module reloaded successfully.")

eva_x module reloaded successfully.


In [ ]:
import torch
import eva_x
from eva_x import eva_x_tiny_patch16
import torch.nn as nn
import torch.optim as optim
from re import X
import timm
import torch.nn.functional as F

class Eva_X_Model(nn.Module):
    def __init__(self):
        super().__init__()
        'img_size = 224, patch_size = 16, embed_dim = 192, depth = 12 transformer layers, num_heads = 3'

        self.eva = eva_x_tiny_patch16(pretrained="/content/drive/MyDrive/chest_xray/eva_x_tiny_patch16_merged520k_mim.pt")
        self.eva.head = nn.Linear(192, 1) #normal vs. abnormal

        #freeze entire Eva
        for p in self.eva.parameters():
            p.requires_grad = False

        #unfreeze last transformer block + norm
        for name, p in self.eva.named_parameters():
            if "head" in name or "blocks.11" in name or "norm" in name or "fc_norm" in name:
                p.requires_grad = True

    def forward(self, x):
        binary_logits = self.eva(x) #returns (B, 1) logits

        return binary_logits

ckpt_path = "/content/drive/MyDrive/chest_xray/checkpoints/eva_x_tiny_binary_best.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Define device here

eva_model = Eva_X_Model()
eva_model.load_state_dict(ckpt["model_state_dict"])
eva_model.to(device)

optimizer = optim.AdamW(eva_model.parameters(), lr=1e-4, weight_decay=1e-4)
optimizer.load_state_dict(ckpt["optimizer_state_dict"])

print("Restored epoch:", ckpt["epoch"] + 1, "best_val_loss:", ckpt["best_val_loss"])
print("class_to_idx:", ckpt.get("class_to_idx"))


_IncompatibleKeys(missing_keys=['head.weight', 'head.bias'], unexpected_keys=[])
Restored epoch: 12 best_val_loss: 0.03127279505133629
class_to_idx: {'NORMAL': 0, 'PNEUMONIA': 1}


In [ ]:
from PIL import Image
import torch
import torchvision.transforms as transforms
import requests
from io import BytesIO

inference_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

def predict_image(source, model, device, threshold=0.5):
    model.eval()

    # Load image from URL or local path
    if source.startswith("http"):
        response = requests.get(source)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(source).convert("RGB")

    image_tensor = inference_transforms(image)
    image_tensor = image_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(image_tensor)
        prob = torch.sigmoid(logits).item()

    prediction = int(prob > threshold)

    return {
        "source": source,
        "p_abnormal": round(prob, 4),
        "prediction": prediction,
        "label_name": "abnormal" if prediction == 1 else "normal"
    }

In [ ]:
#https://radiopaedia.org/cases/pneumonia-7

url = "https://prod-images-static.radiopaedia.org/images/554213/27e6e725de1f349479fd214944aaf0d32523ec38f111107b6f347a3530e4b4ff_big_gallery.jpeg"

result = predict_image(url, eva_model, device)
print(result)

{'source': 'https://prod-images-static.radiopaedia.org/images/554213/27e6e725de1f349479fd214944aaf0d32523ec38f111107b6f347a3530e4b4ff_big_gallery.jpeg', 'p_abnormal': 0.9988, 'prediction': 1, 'label_name': 'abnormal'}


In [ ]:
# Re-define load_image here, as it was in a deleted cell and is needed by diagnose_chest_xray
import requests
from io import BytesIO
from PIL import Image

'''
def load_image(image_file):
    if image_file.startswith('http') or image_file.startswith('https'):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        image = Image.open(image_file).convert('RGB')
    return image
'''

def diagnose_chest_xray(image_path, eva_model, device, threshold=0.5):
    result = predict_image(image_path, eva_model, device) # `predict_image` is global

    # Build base output (always returned)
    out = {
        "source": image_path,
        "p_abnormal": result["p_abnormal"],
        "y_pred": int(result["p_abnormal"] > threshold),
        "reasoning": None
    }

    # If predicted normal, stop early
    if out["p_abnormal"] <= threshold:
        out["reasoning"] = "Classifier predicts NORMAL; reasoning model not invoked."
        return out["reasoning"]

    image = load_image_small(image_path, size=448)  # if OOM -> 336
    #instruction = "Write FINDINGS and IMPRESSION. Be concise. If unsure, say so."

    instruction =  f"""A pretrained vision transformer fine-tuned for binary classification of chest X-rays
(normal vs pneumonia-like abnormality) assigned this image an abnormal probability of
{out['p_abnormal']:.2f}. This classifier does not distinguish among specific pathologies.

Please analyze the image and describe possible radiologic findings that could explain this abnormal signal.
Provide a cautious interpretation and discuss uncertainty.
"""
    return predict_radiology_description(image, instruction)


In [ ]:
path = "/content/drive/MyDrive/chest_xray/train/PNEUMONIA/person1000_bacteria_2931.jpeg"

result = diagnose_chest_xray(path, eva_model, device)

print(result)

user

A pretrained vision transformer fine-tuned for binary classification of chest X-rays
(normal vs pneumonia-like abnormality) assigned this image an abnormal probability of
1.00. This classifier does not distinguish among specific pathologies.

Please analyze the image and describe possible radiologic findings that could explain this abnormal signal.
Provide a cautious interpretation and discuss uncertainty.
assistant

**Step 1: Identify the image**

This is an X-ray of a child's chest. The image shows a large, diffuse opacification involving both lungs, with a few small, well-defined nodules in the right upper lobe. There is no evidence of pneumothorax or pneumopericardium.

**Step 2: Interpretation**

The diffuse opacification is consistent with a diagnosis of respiratory distress syndrome (RDS), also known as hyaline membrane disease. RDS is a condition that affects newborns and is characterized by the accumulation of fluid in the lungs, which can lead to respiratory failure


In [ ]:
# Re-define load_image here, as it was in a deleted cell and is needed by diagnose_chest_xray
import requests
from io import BytesIO
from PIL import Image

'''
def load_image(image_file):
    if image_file.startswith('http') or image_file.startswith('https'):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        image = Image.open(image_file).convert('RGB')
    return image
'''

def diagnose_chest_xray(image_path, eva_model, device, threshold=0.5):
    result = predict_image(image_path, eva_model, device) # `predict_image` is global

    # Build base output (always returned)
    out = {
        "source": image_path,
        "p_abnormal": result["p_abnormal"],
        "y_pred": int(result["p_abnormal"] > threshold),
        "reasoning": None
    }

    # If predicted normal, stop early
    if out["p_abnormal"] <= threshold:
        out["reasoning"] = "Classifier predicts NORMAL; reasoning model not invoked."
        return out["reasoning"]

    image = load_image_small(image_path, size=448)  # if OOM -> 336
    #instruction = "Write FINDINGS and IMPRESSION. Be concise. If unsure, say so."

    instruction = f"""A binary chest X-ray classifier assigned this image an abnormal probability of
    {out['p_abnormal']:.2f}. The classifier only distinguishes normal vs pneumonia-like abnormality.

    As a radiology assistant:

    • First describe observable imaging features.
    • Then provide a brief differential diagnosis list.
    • Indicate the level of certainty (low / moderate / high).
    • Avoid making a definitive medical diagnosis.

    Emphasize that interpretation is preliminary and image-only.
    """
    return predict_radiology_description(image, instruction)


In [ ]:
path = "/content/drive/MyDrive/chest_xray/train/PNEUMONIA/person1000_bacteria_2931.jpeg"

result = diagnose_chest_xray(path, eva_model, device)

print(result)

user

A binary chest X-ray classifier assigned this image an abnormal probability of 
    1.00. The classifier only distinguishes normal vs pneumonia-like abnormality.

    As a radiology assistant:

    • First describe observable imaging features.
    • Then provide a brief differential diagnosis list.
    • Indicate the level of certainty (low / moderate / high).
    • Avoid making a definitive medical diagnosis.

    Emphasize that interpretation is preliminary and image-only.
    assistant

**Step 1: Describe observable imaging features.**

*   The X-ray image shows a dense, hazy appearance in the right upper lung zone.
*   There is an air-bronchogram in the right upper lung zone.
*   The heart is displaced to the left.

**Step 2: Provide a brief differential diagnosis list.**

*   Pneumonia
*   Lung cysts
*   Lung abscess

**Step 3: Indicate the level of certainty.**

*   Moderate

**Step 4: Avoid making a definitive medical diagnosis.**

*   This is a preliminary interpretation 

If p_abnormal > 0.8: ask for likely patterns

If 0.5 < p_abnormal < 0.7: ask for subtle findings and emphasize uncertainty

In [ ]:
def diagnose_chest_xray(image_path, eva_model, device, threshold=0.5):
    result = predict_image(image_path, eva_model, device)

    out = {
        "source": image_path,
        "p_abnormal": result["p_abnormal"],
        "y_pred": int(result["p_abnormal"] > threshold),
        "reasoning": None
    }

    # Normal case
    if out["p_abnormal"] <= threshold:
        out["reasoning"] = "Classifier predicts NORMAL; reasoning model not invoked."
        return out

    image = load_image_small(image_path, size=448)

    p = out["p_abnormal"]

    # High confidence abnormal
    if p > 0.8:
        instruction = f"""
A binary chest X-ray classifier assigned this image an abnormal probability of {p:.2f}.

Given the strong abnormal signal:

• Describe the dominant and most likely radiographic patterns visible.
• Explain which imaging features most strongly support abnormality.
• Provide 2–3 likely explanations.
• Indicate level of certainty.
• Avoid definitive diagnosis.

Focus on prominent findings rather than subtle ones.
"""

    #Borderline abnormal
    elif 0.5 < p <= 0.7:
        instruction = f"""
A binary chest X-ray classifier assigned this image an abnormal probability of {p:.2f}, indicating a borderline abnormal signal.

• Carefully describe any subtle or equivocal imaging findings.
• Discuss alternative normal variants that could mimic abnormality.
• Emphasize uncertainty.
• Provide low-confidence differential considerations only if justified.

Avoid strong conclusions and clearly state limitations.
"""

    #Moderate abnormal (between 0.7 and 0.8)
    else:
        instruction = f"""
A binary chest X-ray classifier assigned this image an abnormal probability of {p:.2f}.

• Describe observable imaging features.
• Provide a short differential diagnosis list.
• Indicate level of certainty.
• Avoid definitive medical diagnosis.
• Emphasize that interpretation is preliminary and image-only.
"""

    out["reasoning"] = predict_radiology_description(image, instruction)
    return out

In [ ]:
path = "/content/drive/MyDrive/chest_xray/train/PNEUMONIA/person1000_bacteria_2931.jpeg"

result = diagnose_chest_xray(path, eva_model, device)

print(result)

{'source': '/content/drive/MyDrive/chest_xray/train/PNEUMONIA/person1000_bacteria_2931.jpeg', 'p_abnormal': 0.9992, 'y_pred': 1, 'reasoning': 'user\n\n\nA binary chest X-ray classifier assigned this image an abnormal probability of 1.00.\n\nGiven the strong abnormal signal:\n\n• Describe the dominant and most likely radiographic patterns visible.\n• Explain which imaging features most strongly support abnormality.\n• Provide 2–3 likely explanations.\n• Indicate level of certainty.\n• Avoid definitive diagnosis.\n\nFocus on prominent findings rather than subtle ones.\nassistant\n\nThe provided chest X-ray image is a binary image, indicating that it has been processed to enhance the visibility of certain features while reducing others. This processing can make it easier to identify certain patterns and features.\n\n**Dominant and Most Likely Radiographic Patterns Visible:**\n\n*   **Increased Lung Density:** The most striking feature in this image is the increased density of the lungs, w

In [ ]:
path = "/content/drive/MyDrive/chest_xray/train/NORMAL/IM-0115-0001.jpeg"

result = diagnose_chest_xray(path, eva_model, device)

print(result)